# Track Condition — Train & Export ONNX (Colab)

Fine-tune a small, fast **MobileNetV3** to classify a single frame as **Dry / Damp / Wet**,
then export it to **`model.onnx`** so it drops straight into the `inference/` service of the app.

> Reminder: a single frame can only show the *moisture level*. "Drying" is detected later in the
> app's backend from how the level moves over time — you do **not** train a "Drying" class here.

**How to use:** Runtime → *Change runtime type* → **GPU**, then run each cell top to bottom.
At the end you download `model.onnx` + `class_names.json` and copy them into `inference/model/`.


## 1. Install / import

In [ ]:
# Colab already has torch + torchvision. We only need onnx tooling.
!pip -q install onnx onnxruntime

import os, json, time, shutil, random
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 2. Get a dataset

You need images sorted into folders by class. The app expects **Dry / Damp / Wet**.

**Option A — Use your own frames (best).** Upload a zip laid out like this and unzip to `/content/data`:
```
data/
  train/Dry/*.jpg   train/Damp/*.jpg   train/Wet/*.jpg
  val/Dry/*.jpg     val/Damp/*.jpg     val/Wet/*.jpg
```

**Option B — Public datasets** (see repo README for links): RoadSaW (dry/damp/wet/very-wet — merge
`very wet`→`Wet`), or RSCD. Download, then re-arrange into the folder layout above.

**Option C — Quick smoke test with synthetic data** (below) just to prove the pipeline end-to-end.
Replace it with real data before trusting any accuracy number.


In [ ]:
# ---- Option C: synthetic smoke-test dataset (DELETE once you have real images) ----
from PIL import Image
import numpy as np

ROOT = "/content/data"
CLASSES = ["Dry", "Damp", "Wet"]
BRIGHT = {"Dry": 200, "Damp": 120, "Wet": 60}   # wetter = darker, crude but separable

def make(split, n):
    for c in CLASSES:
        d = f"{ROOT}/{split}/{c}"; os.makedirs(d, exist_ok=True)
        for i in range(n):
            base = BRIGHT[c] + np.random.randint(-25, 25)
            arr = np.clip(np.full((224,224,3), base) + np.random.randint(-15,15,(224,224,3)), 0,255).astype("uint8")
            if c == "Wet":  # specular highlights
                arr[np.random.randint(0,200):][:8, np.random.randint(0,180):][:60] = 255
            Image.fromarray(arr).save(f"{d}/{split}_{c}_{i}.jpg")

if not os.path.exists(f"{ROOT}/train"):
    make("train", 120); make("val", 30)
    print("synthetic dataset created at", ROOT)
else:
    print("dataset already present at", ROOT)


## 3. Data loaders

In [ ]:
IMG = 224
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(f"{ROOT}/train", transform=train_tf)
val_ds   = datasets.ImageFolder(f"{ROOT}/val",   transform=val_tf)

# IMPORTANT: this is the class order the ONNX model will output. Save it for the app.
CLASS_NAMES = train_ds.classes
print("classes (output order):", CLASS_NAMES)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)


## 4. Model — MobileNetV3-Small with a fresh 3-class head (transfer learning)

In [ ]:
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
in_f = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_f, len(CLASS_NAMES))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


## 5. Train

In [ ]:
EPOCHS = 8  # bump to 20-30 on a real dataset

def run_epoch(dl, train):
    model.train() if train else model.eval()
    tot, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train: optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward(); optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            tot      += x.size(0)
    return loss_sum/tot, correct/tot

for ep in range(1, EPOCHS+1):
    tl, ta = run_epoch(train_dl, True)
    vl, va = run_epoch(val_dl, False)
    print(f"epoch {ep:2}/{EPOCHS}  train_acc={ta:.3f}  val_acc={va:.3f}")


## 6. Confusion matrix — watch Damp↔Wet especially

In [ ]:
import numpy as np
model.eval()
n = len(CLASS_NAMES); cm = np.zeros((n,n), int)
with torch.no_grad():
    for x,y in val_dl:
        p = model(x.to(DEVICE)).argmax(1).cpu().numpy()
        for t,pr in zip(y.numpy(), p): cm[t,pr]+=1
print("rows=true, cols=pred |", CLASS_NAMES)
print(cm)


## 7. Export to ONNX

In [ ]:
model.eval().cpu()
dummy = torch.randn(1, 3, IMG, IMG)
torch.onnx.export(
    model, dummy, "model.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13,
)
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)
print("wrote model.onnx and class_names.json")

# sanity-check the exported model runs in onnxruntime
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
out = sess.run(None, {"input": dummy.numpy()})[0]
print("onnx output shape:", out.shape, "->", CLASS_NAMES)


## 8. Download and install into the app

Run the cell, then copy both files into your repo at **`inference/model/`** and set
`MODEL_BACKEND=onnx` in `.env`. Rebuild: `docker compose up --build`. Done — the app now uses
your trained model with **no code changes**.


In [ ]:
from google.colab import files
files.download("model.onnx")
files.download("class_names.json")
